# Play Together — YOLO train on Kaggle

**Setup trước khi Run All:**
1. Upload dataset ZIP lên Kaggle Dataset (hoặc Add Input dataset đã có)
2. Settings → Accelerator → **GPU T4 / P100**
3. Internet: **On**
4. Sửa `DATASET_DIR` ở cell dưới cho đúng đường dẫn Input

In [ ]:
# ===== CONFIG =====
DATASET_DIR = "/kaggle/input/play-together2"  # đổi nếu tên dataset khác
EPOCHS = 50
IMGSZ = 640
BATCH = 16
MODEL = "yolov8n.pt"  # hoặc yolov8s.pt nếu muốn mạnh hơn
RUN_NAME = "playtogether"

In [ ]:
!pip install -q ultralytics

import torch
from pathlib import Path
import yaml

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
root = Path(DATASET_DIR)
assert root.exists(), f"Không thấy {root}. Add dataset rồi sửa DATASET_DIR."

# Tìm data.yaml (có thể nằm ở root hoặc 1 cấp con)
candidates = list(root.rglob("data.yaml"))
assert candidates, f"Không thấy data.yaml trong {root}"
data_yaml = candidates[0]
print("data.yaml:", data_yaml)

# Sửa path train/val cho đúng trên Kaggle
cfg = yaml.safe_load(data_yaml.read_text())
base = data_yaml.parent

def resolve_split(key: str) -> str:
    # chấp nhận train/images hoặc ../train/images
    rel = cfg.get(key) or cfg.get("val" if key == "val" else key)
    if key == "val" and not cfg.get("val"):
        rel = cfg.get("valid")
    p = (base / str(rel).replace("../", "")).resolve()
    if not p.exists():
        # fallback phổ biến
        alt = base / ("valid/images" if key == "val" else f"{key}/images")
        p = alt.resolve()
    assert p.exists(), f"Missing {key}: tried {p}"
    return str(p)

kaggle_yaml = Path("/kaggle/working/data.yaml")
out = {
    "train": resolve_split("train"),
    "val": resolve_split("val"),
    "nc": cfg.get("nc", len(cfg.get("names", []))),
    "names": cfg.get("names"),
}
if cfg.get("test"):
    try:
        out["test"] = resolve_split("test")
    except AssertionError:
        pass

kaggle_yaml.write_text(yaml.dump(out, sort_keys=False))
print(kaggle_yaml.read_text())
print("train images:", len(list(Path(out["train"]).glob("*.*"))))
print("val images:", len(list(Path(out["val"]).glob("*.*"))))

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
results = model.train(
    data=str(kaggle_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    workers=2,
    project="/kaggle/working/runs",
    name=RUN_NAME,
    exist_ok=True,
)
print("save_dir:", results.save_dir)

In [ ]:
from IPython.display import FileLink, display
from pathlib import Path
import shutil

best = Path(results.save_dir) / "weights" / "best.pt"
out = Path("/kaggle/working/best.pt")
shutil.copy2(best, out)
print("best.pt size MB:", round(out.stat().st_size / 1e6, 2))
display(FileLink(str(out)))

# Copy vào máy: bấm link best.pt → tải về → bỏ vào models/best.pt local